In [1]:
import pandas as pd
import numpy as np
from gensim.test.utils import datapath
from gensim import utils
from numba import njit, prange
from numba.typed import List
import pickle

#### Loading `tisch` and `sept` into sentences

In [2]:
sept = pd.read_pickle("./pickles/sept.pickle")
tisch = pd.read_pickle("./pickles/tisch.pickle")

In [3]:
sept_verses = list(
    sept.groupby(["book", "chapter", "verse"])["text"]
    .apply(lambda words: " ".join(words).split())
    .values
)
tisch_verses = list(
    tisch.groupby(["book", "chapter", "verse"])["text"]
    .apply(lambda words: " ".join(words).split())
    .values
)

In [4]:
print("sept verses", sept_verses[:5])
print("tisch verses", tisch_verses[:5])

sept verses [['εν', 'αρχη', 'εποιησεν', 'ο', 'θεος', 'τον', 'ουρανον', 'και', 'την', 'γην'], ['η', 'δε', 'γη', 'ην', 'αορατος', 'και', 'ακατασκευαστος', 'και', 'σκοτος', 'επανω', 'της', 'αβυσσου', 'και', 'πνευμα', 'θεου', 'επεφερετο', 'επανω', 'του', 'υδατος'], ['και', 'ειπεν', 'ο', 'θεος', 'γενηθητω', 'φως', 'και', 'εγενετο', 'φως'], ['και', 'ειδεν', 'ο', 'θεος', 'το', 'φως', 'οτι', 'καλον', 'και', 'διεχωρισεν', 'ο', 'θεος', 'ανα', 'μεσον', 'του', 'φωτος', 'και', 'ανα', 'μεσον', 'του', 'σκοτους'], ['και', 'εκαλεσεν', 'ο', 'θεος', 'το', 'φως', 'ημεραν', 'και', 'το', 'σκοτος', 'εκαλεσεν', 'νυκτα', 'και', 'εγενετο', 'εσπερα', 'και', 'εγενετο', 'πρωι', 'ημερα', 'μια']]
tisch verses [['βίβλος', 'γενέσεως', 'ἰησοῦ', 'χριστοῦ', 'υἱοῦ', 'δαυεὶδ', 'υἱοῦ', 'ἀβραάμ'], ['ἀβραὰμ', 'ἐγέννησεν', 'τὸν', 'ἰσαάκ', 'ἰσαὰκ', 'δὲ', 'ἐγέννησεν', 'τὸν', 'ἰακώβ', 'ἰακὼβ', 'δὲ', 'ἐγέννησεν', 'τὸν', 'ἰούδαν', 'καὶ', 'τοὺς', 'ἀδελφοὺς', 'αὐτοῦ'], ['ἰούδας', 'δὲ', 'ἐγέννησεν', 'τὸν', 'φάρες', 'καὶ', 'τὸν', 'ζάρα

#### Book, verse, chapter tisch and sept db

In [5]:
sept_plain = sept.drop(["rmac", "str"], axis="columns")
tisch_plain = tisch.drop(["rmac", "str"], axis="columns")
print("Sept, then tisch.")
display(sept_plain.head(2))
display(tisch_plain.head(2))

Sept, then tisch.


,text,verse,chapter,book
0,εν,1,1,1
1,αρχη,1,1,1


,text,verse,chapter,book
0,βίβλος,1,1,40
1,γενέσεως,1,1,40


## Making the word2vec Model

In [6]:
import gensim.models

model = gensim.models.Word2Vec(
    sentences=sept_verses + tisch_verses,
    min_count=1,
)

In [7]:
try:
    model = gensim.models.Word2Vec.load("word2vec.model")
except FileNotFoundError:
    model = gensim.models.Word2Vec(vector_size=100, window=5, min_count=1, workers=4)
    model.build_vocab(sept_verses + tisch_verses)
    model.train(
        sept_verses + tisch_verses,
        total_examples=model.corpus_count,
        epochs=100,
        compute_loss=True,
    )
    model.save("word2vec.model")

In [8]:
model.wv.most_similar("θεος")[:3]

[('κυριος', 0.6227647066116333),
 ('πατηρ', 0.5973913073539734),
 ('βασιλευς', 0.5650644898414612)]

In [9]:
print(model.wv.index_to_key[:4])

['και', 'εν', 'καὶ', 'του']


In [10]:
matt_opening = tisch.query("book == 40")[0:5]
luke_opening = tisch.query("book == 41")[0:5]
mark_opening = tisch.query("book == 42")[0:5]

### Making the Sentence comparision method

In [11]:
np.linalg.norm(model.wv["καὶ"] - model.wv["τοῦ"])

27.3983

In [12]:
model.wv.similarity("καὶ", "τοῦ")

0.66204125

In [13]:
@njit
def word_vec_distance(verse1: list[np.array], verse2: list[np.array]) -> float:
    minimum_distances_1 = []
    for word1 in verse1:
        each_word = []
        for word2 in verse2:
            each_word.append(np.linalg.norm(word1 - word2))
        minimum_distances_1.append(min(each_word))

    minimum_distances_2 = []
    for word2 in verse2:
        each_word = []
        for word1 in verse1:
            each_word.append(np.linalg.norm(word2 - word1))
        minimum_distances_2.append(min(each_word))
    
    average_1 = sum(minimum_distances_1)/len(minimum_distances_1)
    average_2 = sum(minimum_distances_2)/len(minimum_distances_2)
    return (average_1 + average_2) / 2

In [14]:
@njit(parallel=True)
def find_versewise_similarity(sept_verse_vecs, tisch_verse_vecs):
    ot_len = len(sept_verse_vecs)
    nt_len = len(tisch_verse_vecs)
    data = np.empty((ot_len, nt_len), dtype=float)
    for nidx in prange(len(tisch_verse_vecs)):
        nverse = tisch_verse_vecs[nidx]
        for oidx, overse in enumerate(sept_verse_vecs):
            data[oidx, nidx] = word_vec_distance(overse, nverse)
    return data

In [15]:
def to_numba_list(nested):
    doc = List()
    for verse in nested:
        nverse = List()
        for word in verse:
            nverse.append(word)
        doc.append(nverse)
    return doc

In [16]:
with open('pickles/sept_verse_vecs.pickle', 'rb') as f:
    sept_verse_vecs = to_numba_list(pickle.load(f))
with open('pickles/tisch_verse_vecs.pickle', 'rb') as f:
    tisch_verse_vecs = to_numba_list(pickle.load(f))

In [17]:
similarities = find_versewise_similarity(sept_verse_vecs, tisch_verse_vecs)
with open('similarities.pickle', 'wb') as f:
    pickle.dump(similarities, f)

/tmp/ipykernel_15029/1697069189.py:7: NumbaTypeSafetyWarning: unsafe cast from uint64 to int64. Precision may be lost.
  nverse = tisch_verse_vecs[nidx]


In [18]:
similarities.shape

(22842, 7940)